# Requirements & Reproducibility

## Necessary Libraries

In [ ]:
!pip install -q rapidfuzz evaluate seqeval

print("\nSUCCESS: Libraries installed")

In [ ]:
import evaluate
import html
import importlib.metadata
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import random
import re
import requests
import seaborn as sns
import sys
import time
import torch
import unicodedata

from collections import defaultdict, Counter
from datasets import Dataset
from google.colab import userdata, files
from openai import OpenAI, RateLimitError
from rapidfuzz import fuzz
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    TrainingArguments,
    Trainer
)
from transformers import pipeline as hf_pipeline
from urllib.parse import quote

print("SUCCESS: Necessary packages imported")

## Versions

In [ ]:
packages = [
    'matplotlib', 'numpy', 'pandas', 'requests', 'seaborn',
    'torch', 'datasets', 'openai', 'rapidfuzz', 'sklearn',
    'transformers', 'evaluate', 'seqeval'
]

print(f"Python version: {sys.version.split()[0]}")
for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package}: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\nSUCCESS: All versions verified")

## Random Seeds

In [ ]:
GLOBAL_SEED = 42
os.environ['PYTHONHASHSEED'] = str(GLOBAL_SEED)
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.cuda.manual_seed(GLOBAL_SEED)
torch.cuda.manual_seed_all(GLOBAL_SEED) # for multi-GPU
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("SUCCESS: Random seeds set")

# DPLA Record Ingestion

## Ingestion Pipeline

In [ ]:
# =========================================================================================================================================
# Configuration
# =========================================================================================================================================

OUTPUT_FILE = "rawIngestedMetadata.json"

COHORT_CONFIG = {
    "Cohort A (Racial/Ethnic Minorities)": [
        '"Civil Rights" OR "Civil Rights Movement"',
        '"Racial segregation" OR "Jim Crow" OR "Desegregation"',
        '"African American history" OR "African-American history" OR "Black history"',
        '"Chicano movement" OR "Chicana movement" OR "El Movimiento"',
        '"Asian American history" OR "Asian-American history" OR "Asian American"'
    ],
    "Cohort B (LGBTQIA+ Histories)": [
        '"Transgender" OR "Transsexual" OR "Gender identity" OR "Gender nonconforming" OR "Genderqueer" OR "Transgender people"',
        '"LGBTQ history" OR "LGBT history" OR "Gay and lesbian history"',
        '"Queer oral history" OR "LGBTQ oral history" OR "Gay oral history"',
        '"Stonewall riots" OR "Stonewall uprising" OR "Stonewall Rebellion"',
        '"Gay liberation movement" OR "Gay Liberation Front" OR "Gay rights movement"'
    ],
    "Cohort C (Indigenous Populations)": [
        '"Indigenous peoples of North America" OR "Native American" OR "American Indian"',
        '"Red Power movement" OR "Red Power" OR "American Indian Movement"',
        '"Navajo Indians" OR "Navajo Nation" OR "Diné" OR "Dineh" OR "Navajo people" OR "Navajo reservation"',
        '"Tribal sovereignty" OR "Native American treaties" OR "Indian treaties" OR "Tribal government" OR "Indigenous land rights" OR "Indian self-determination"',
        '"Trail of Tears" OR "Indian Removal Act"'
    ]
}

combined_records = [] # sets up list for successfully extracted records

MAX_RECORDS_PER_SUBQUERY = 100
RECORDS_PER_PAGE_DPLA = 20
MAX_PAGES_PER_QUERY = 10

# tracks distribution of ingested records
cohort_counts = {cohort: 0 for cohort in COHORT_CONFIG.keys()}
subquery_distribution = {}

seen_ids = set() # sets up registry to prevent duplicated records if subquery results overlap

try:
    DPLA_API_KEY = userdata.get('DPLA_API_KEY')
except Exception:
    DPLA_API_KEY = None

if not DPLA_API_KEY:
    raise ValueError("DPLA_API_KEY NOT FOUND.")

# =========================================================================================================================================
# Text-parsing and Normalization
# =========================================================================================================================================

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f\x7f-\x9f]', '', text)
    text = text.replace('\xad', '')
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def safe_stringify(field_data):
    if not field_data:
        return ""
    if isinstance(field_data, list):
        return " ".join([str(item) for item in field_data if item])
    return str(field_data)

# =========================================================================================================================================
# DPLA Extraction Loop
# =========================================================================================================================================

session = requests.Session()
session.headers.update({
    "User-Agent": f"DSA598CapstoneProjectRecordIngestion/1.0 (AutomatedIngestion; Seed=42)",
    "Accept": "application/json"
})

print(f"Initializing DPLA Ingestion Pipeline.")
print(f"Cap Strategy: Max {MAX_RECORDS_PER_SUBQUERY} records per individual keyword.")

dpla_url = "https://api.dp.la/v2/items"
DESC_CHAR_LIMIT = 1500 # sets limit for characters in description

for cohort, queries in COHORT_CONFIG.items():
    print(f"\nSegmenting Cohort Domain: {cohort}")
    subquery_distribution[cohort] = {}

    for query in queries:
        query_harvested = 0
        subquery_distribution[cohort][query] = 0
        print(f"Ingesting subquery string: '{query}'")

        for page in range(1, MAX_PAGES_PER_QUERY + 1):
            if query_harvested >= MAX_RECORDS_PER_SUBQUERY:
                break

            dpla_params = {
                "q": query,
                "api_key": DPLA_API_KEY,
                "page_size": RECORDS_PER_PAGE_DPLA,
                "page": page
            }

            max_retries = 5

            for attempt in range(max_retries):
              response = requests.get(dpla_url, params=dpla_params)
              if response.status_code == 200:
                break
              elif response.status_code == 429:
                sleep_time = 2 ** (attempt + 1)
                print(f"Rate limited on '{query}'. Retrying in {sleep_time}s...")
                time.sleep(sleep_time)
              else:
                print(f"API Error {response.status_code} for '{query}'")
                break

            dpla_data = response.json()
            docs = dpla_data.get("docs", [])

            if not docs:
                print(f"Page {page}: Natural completion (Index fully depleted for '{query}').")
                break

            for doc in docs:
                if query_harvested >= MAX_RECORDS_PER_SUBQUERY:
                    break

                try:
                    local_id = safe_stringify(doc.get("id", "N/A"))

                    if local_id in seen_ids or local_id == "N/A":
                        continue

                    source_resource = doc.get("sourceResource", {})
                    title_text = safe_stringify(source_resource.get("title", "Untitled Record"))

                    raw_description = source_resource.get("description", [])

                    if isinstance(raw_description, list):
                        description_text = " ".join([safe_stringify(d) for d in raw_description])
                    else:
                        description_text = safe_stringify(raw_description)

                    clean_title = clean_text(title_text)
                    clean_desc = clean_text(description_text)

                    # skip records without a substantive description
                    if not clean_desc.strip() or len(clean_desc.split()) < 5:
                        continue

                    # truncates overlong records without slicing words
                    if len(clean_desc) > DESC_CHAR_LIMIT:
                        truncated = clean_desc[:DESC_CHAR_LIMIT]
                        last_space = truncated.rfind(' ')
                        if last_space > 0:
                            clean_desc = truncated[:last_space] + "..."
                        else:
                            clean_desc = truncated + "..."

                    flat_record = {
                        "local_id": local_id,
                        "source": "Digital Public Library of America",
                        "cohort": cohort,
                        "theme": clean_text(query),
                        "title": clean_title,
                        "description": clean_desc
                    }

                    combined_records.append(flat_record) # adds flattened JSON for successfully extracted items
                    seen_ids.add(local_id) # adds successfully extracted items to registry to avoid duplication

                    cohort_counts[cohort] += 1
                    query_harvested += 1
                    subquery_distribution[cohort][query] += 1

                except Exception:
                    continue

        print(f"Total extracted for '{query}': {subquery_distribution[cohort][query]} / {MAX_RECORDS_PER_SUBQUERY} records.")

# =========================================================================================================================================
# Output Serialization
# =========================================================================================================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(combined_records, f, indent=4, ensure_ascii=False)

print(f"\nSUCCESS: DPLA Record Ingestion Complete. Total Unique Validated Records Preserved: {len(combined_records)} and raw JSON file saved to: {os.path.abspath(OUTPUT_FILE)}.")

# =========================================================================================================================================
# Data Quality Evaluation
# =========================================================================================================================================

def generate_quality_metrics_with_plots(records: list, desc_limit: int = 1500):
    if not records:
        print("No records available to analyze.")
        return

    df = pd.DataFrame(records)

    df['title_char_count'] = df['title'].apply(len)
    df['desc_char_count'] = df['description'].apply(len)
    df['desc_word_count'] = df['description'].apply(lambda x: len(x.split()))

    # prints to console
    print("\n\nDATA QUALITY METRICS::")

    print("\nCOHORT BALANCE")
    cohort_counts = df['cohort'].value_counts()
    for cohort, count in cohort_counts.items():
        percentage = (count / len(df)) * 100
        print(f"{cohort}: {count} records ({percentage:.1f}%)")

    print("\nTEXT DENSITY")
    print("Description Word Count:")
    print(f"  Mean:   {df['desc_word_count'].mean():.1f} words")
    print(f"  Median: {df['desc_word_count'].median():.1f} words")
    print(f"  Min:    {df['desc_word_count'].min()} words")
    print(f"  Max:    {df['desc_word_count'].max()} words")

    print("\nTRUNCATED RECORDS")
    # checks for lengths near description character limit
    truncated_df = df[df['desc_char_count'] >= (desc_limit - 10)]
    trunc_rate = (len(truncated_df) / len(df)) * 100
    print(f"Descriptions truncated at limit: {len(truncated_df)} ({trunc_rate:.1f}%)")

    print("\nMETADATA QUALITY")
    untitled_count = len(df[df['title'] == "Untitled Record"])
    print(f"Missing Titles ('Untitled Record'): {untitled_count} ({(untitled_count/len(df))*100:.1f}%)")

    # viusalized data quality metrics
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Plot 1: Cohort Distribution
    sns.countplot(
        data=df,
        x='cohort',
        hue='cohort',
        order=df['cohort'].value_counts().index,
        palette='viridis',
        legend=False,
        ax=axes[0]
    )
    axes[0].set_title("Distribution of Records by Cohort", fontsize=14, fontweight='bold')
    axes[0].set_xlabel("Cohort Domain", fontsize=12)
    axes[0].set_ylabel("Number of Records", fontsize=12)
    axes[0].tick_params(axis='x', rotation=45)

    # Plot 2: Text Density Histogram
    sns.histplot(
        data=df,
        x='desc_word_count',
        bins=30,
        kde=True,
        color='coral',
        ax=axes[1]
    )
    axes[1].set_title("Description Word Count Density", fontsize=14, fontweight='bold')
    axes[1].set_xlabel("Word Count", fontsize=12)
    axes[1].set_ylabel("Frequency", fontsize=12)

    plt.tight_layout()
    plt.show()

generate_quality_metrics_with_plots(combined_records, desc_limit=1500)

## Manually Annotated (Gold-Standard) Records

### Manually Annotated (Gold-Standard) Record Selection

In [ ]:
# =========================================================================================================================================
# Configuration
# =========================================================================================================================================

INPUT_FILE = "rawIngestedMetadata.json"
OUTPUT_FILE = "manualAnnotationTemplate.json"
RECORDS_PER_COHORT = 25

MIN_WORDS = 20
MAX_CHARS = 1500

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Master dataset file '{INPUT_FILE}' not found.")

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    master_records = json.load(f)

cohort_buckets = {} # stratifying records by cohort for disaggregated evaluation
for record in master_records:
    cohort = record["cohort"]
    if cohort not in cohort_buckets:
        cohort_buckets[cohort] = []
    cohort_buckets[cohort].append(record)

gold_standard_records = [] # setting up list for selections

# =========================================================================================================================================
# Record Selection Loop
# =========================================================================================================================================

print("Commencing Stratified Evaluation Selection...")
for cohort, records in cohort_buckets.items():
    viable_candidates = [
        r for r in records
        if len(r["description"].split()) >= MIN_WORDS
        and len(r["description"]) <= MAX_CHARS
    ]

    if len(viable_candidates) < RECORDS_PER_COHORT:
        print(f"Warning: Group [{cohort}] only has {len(viable_candidates)} candidates fitting the {MAX_CHARS}-char limit. Expanding search...")
        viable_candidates = [
            r for r in records
            if len(r["description"].split()) >= MIN_WORDS
        ]

    sample_size = min(len(viable_candidates), RECORDS_PER_COHORT)
    cohort_sample = random.sample(viable_candidates, sample_size)

    for item in cohort_sample:
        annotated_entry = {
            "local_id": item["local_id"],
            "cohort": item["cohort"],
            "theme": item["theme"],
            "title": item["title"], # maintained for manual annotator
            "description" : item["description"], # text to be annotated
            "entities": [{
                "start": 0,
                "end": 0,
                "label": "",
                "entity_span": "",
                "wikidata_id": ""
            }]
        }

        gold_standard_records.append(annotated_entry)

    print(f"Successfully sampled {len(cohort_sample)} items from [{cohort}].")

# =========================================================================================================================================
# Output Serialization
# =========================================================================================================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(gold_standard_records, f, indent=4, ensure_ascii=False)

print(f"\nTotal Cohort Target Selection Size: {len(gold_standard_records)} items.")
print(f"SUCCESS: Manual Annotation (Gold-Standard) Template Frozen. File generated for manual annotation: {OUTPUT_FILE}")

### Manual Annotation (Gold-Standard) Guide

Manual labeling completed in enterprise version of [Label Studio](https://https://labelstud.io/) (version 1.23.0) using the following setup:

```xml
<View><View>
  <!-- headers for context -->
  <Header value="Title: $title" size="3" />
  <Header value="Cohort: $cohort | Theme: $theme" size="5" style="color: #666; margin-bottom: 15px;" />
  
  <!-- text to be annotated -->
  <Text name="text" value="$description" />


  <!-- Named Entity Recognition Labels -->
  <Labels name="label" toName="text">
    <Label value="PERSON" background="#FF4D4D"/>
    <Label value="EVENT" background="#F1C40F"/>
    <Label value="GPE" background="#E67E22"/>
    <Label value="ORG" background="#3399FF"/>
    <Label value="NORP" background="#9B59B6"/>
  </Labels>
</View></View>
```

#### Entity Tag Guidelines:

PERSON (Named Individuals)
```
- Definition: Specific, named historical figures, individuals, or well-known aliases.
- Inclusions: Full names, surnames used alone, middle initials, or universally recognized historical pseudonyms.
- Boundaries: Highlight the name itself. Do not include professional, political, or familial titles (e.g., Dr., President, Chief, Mom, Uncle) or possessive suffixes (e.g., 's).
- Examples:
  - "President [Lincoln] signed the order."
  - "The activism of [Marsha P. Johnson]'s contemporaries..."
  - "A performance by Mom and Pop [Winans] drew a crowd."
```
ORG (Organizations & Institutions)
```
- Definition: Formally structured bodies, political organizations, activist groups, government agencies, schools, or businesses.
- Inclusions: Full formal names, universally understood acronyms, operational branches, and family-owned businesses (e.g., Tom’s Bagels acting as an incorporated entity).
- Examples:
  - "The [Student Nonviolent Coordinating Committee] mobilized voters."
  - "A directive from the [Bureau of Indian Affairs]."
  - "They bought their coffee at [Tom’s Bagels]."
```
GPE (Geopolitical Entities)
```
- Definition: Physical locations that possess a recognized government, administrative boundary, or legal political structure.
- Inclusions: Cities, states, countries, recognized reservations, and sovereign tribal nations.
- Examples:
  - "Protests erupted across [Alabama]."
  - "A delegation traveled to [Washington D.C.]"
```

EVENT (Named Historical Occurrences)
```
- Definition: Specific named historical events, battles, marches, riots, strikes, formalized socio-political movements, and organized gatherings like concerts or festivals.
- Inclusions: The full proper name of the event, movement, or performance gathering.
- Examples:
  - "The catalyst of the [Stonewall Riots] changed the landscape."
  - "Thousands joined the [March on Washington]."
  - "She performed at the [Woodstock Music Festival]."
```
NORP (Nationalities, Religious, or Political Groups)
```
- Definition: Named groups of people sharing a common cultural identity, ethnicity, nationality, religious affiliation, or political alignment.
- Inclusions: Cultural group names, indigenous tribal affiliations, and collective community descriptors.
- Examples:
  - "Organizing opportunities for [Chicano] farm workers."
  - "The history of the [Diné] people."
```

#### General Guidelines:

**Multiple Mentions:**

Tag an entity every single time it appears in a description. If a name or group appears five times in one text block, highlight it five times. Leaving later instances untagged accidentally trains the model that those words are normal text.

**Contextual Isolation:**

Evaluate each sentence strictly on its own merits. If an individual is introduced by full name in sentence one ([Martin Luther King Jr.] → PERSON) and referred to only by last name in sentence three, tag the last name independently ([King] → PERSON).

**Overlapping Tokens:**

A single word cannot belong to two categories simultaneously. Choose the category that reflects the highest specific order of context in that sentence (e.g., tag "The 1960 Black Panther rally" entirely as EVENT, rather than breaking "Black Panther" out into an ORG).

**Acronyms & Parentheses:**

When an organization is listed as a full name followed by its acronym in parentheses, tag them as two separate, independent ORG entities. Do not highlight the parentheses themselves.


### Reconciling Label Studio Output

In [ ]:
# =========================================================================================================================================
# Merging Label Studio Annotations with manualAnnotationTemplate.json
# =========================================================================================================================================

TEMPLATE = 'manualAnnotationTemplate.json'
ANNOTATIONS = 'annotations.json' # file containing labeled data
OUTPUT_FILE = 'manualAnnotationFinal.json'

with open(TEMPLATE, 'r', encoding='utf-8') as f:
    template_data = json.load(f)

with open(ANNOTATIONS, 'r', encoding='utf-8') as f:
    annotated_data = json.load(f)

# dictionary mapping the local_id to the extracted entities
entity_map = {}
for item in annotated_data:
    local_id = item.get('data', {}).get('local_id')
    if not local_id:
        continue

    formatted_entities = []

    # extract annotations
    annotations = item.get('annotations', [])
    if annotations:
        results = annotations[0].get('result', [])

        for res in results:
            value = res.get('value', {})
            if not value:
                continue

            # specific fields
            start = value.get('start', 0)
            end = value.get('end', 0)
            entity_span = value.get('text', '')
            labels = value.get('labels', [])
            label = labels[0] if labels else ""

            # reformat
            formatted_entities.append({
                "start": start,
                "end": end,
                "label": label,
                "entity_span": entity_span,
                "wikidata_id": "" # still blank for now
            })

    if formatted_entities:
        entity_map[local_id] = formatted_entities

# update template file
updated_count = 0
for item in template_data:
    local_id = item.get('local_id')

    if local_id in entity_map:
        item['entities'] = entity_map[local_id]
        updated_count += 1

# =========================================================================================================================================
# Output Serialization
# =========================================================================================================================================

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(template_data, f, indent=4)

print(f"Successfully updated {updated_count} records.")
print(f"File saved to: {OUTPUT_FILE}")

#### Example Record:

```json
{
    "local_id": "dpla_example_id_123",
    "cohort": "Cohort B (LGBTQIA+ Histories)",
    "theme": "Stonewall riots",
    "title": "Photo of Sylvia Rivera",
    "text": "In June 1969, Sylvia Rivera and other activists resisted a police raid at the Stonewall Inn in New York City.",
    "entities": [
        {
            "start": 14,
            "end": 27,
            "label": "PERSON",
            "entity_span": "Sylvia Rivera",
            "wikidata_id": "Q465671" # added into manualAnnotationFinal.json manually
        },
        ...
        {
            "start": 95,
            "end": 108,
            "label": GPE,
            "entity_span": "New York City",
            "wikidata_id": "Q60"
        },
    ]
}
```

# Named Entity Recognition

#### NER Model: Phase 1

In [ ]:
# =========================================================================================================================================
# Configuration
# =========================================================================================================================================

INPUT_FILE = "manualAnnotationFinal.json"
GLOBAL_SEED = 42

TARGET_LABELS = {"PERSON", "ORG", "GPE", "EVENT", "NORP"}

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    gold_data = json.load(f)

 # setting up custom IDs
label_list = ["O"]
for label in TARGET_LABELS:
    label_list.extend([f"B-{label}", f"I-{label}"])

label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for label, i in label_to_id.items()}

# =========================================================================================================================================
# Data Preparation
# =========================================================================================================================================

def convert_to_hf_dataset(records, tokenizer, max_length=512):
    dataset_rows = {"input_ids": [], "attention_mask": [], "offset_mapping": [], "labels": []}

    for record in records:
        text = record["description"]
        entities = record.get("entities", [])

        tokenized = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_offsets_mapping=True
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]
        offsets = tokenized["offset_mapping"]

        # initialize tags with -100 for special tokens, "O" for text
        ner_tags = []
        for start, end in offsets:
            if start == end:
                ner_tags.append(-100)
            else:
                ner_tags.append(label_to_id["O"])

        for idx, (start, end) in enumerate(offsets):
            if start == end: continue

            for entity in entities:
                e_start, e_end, label = entity["start"], entity["end"], entity["label"]

                if label not in TARGET_LABELS:
                    continue

                if start <= e_start < end:
                    tag_str = f"B-{label}"
                    if tag_str in label_to_id:
                        ner_tags[idx] = label_to_id[tag_str]
                elif start < e_end and end > e_start:
                    tag_str = f"I-{label}"
                    if tag_str in label_to_id:
                        ner_tags[idx] = label_to_id[tag_str]

        dataset_rows["input_ids"].append(input_ids)
        dataset_rows["attention_mask"].append(attention_mask)
        dataset_rows["offset_mapping"].append(offsets)
        dataset_rows["labels"].append(ner_tags)

    return Dataset.from_dict(dataset_rows)

cohort_stratify_labels = [record.get("cohort", "Unknown Cohort").strip() for record in gold_data]

train_records, test_records = train_test_split(
        gold_data,
        test_size=0.4,
        stratify=cohort_stratify_labels,
        random_state=GLOBAL_SEED
)

print("\nSPLIT SUMMARY")
print(f"Total Gold Standard Records: {len(gold_data)}")
print(f" -> Balanced Training Pool:   {len(train_records)} records")
print(f" -> Balanced Evaluation Pool: {len(test_records)} records")

# =========================================================================================================================================
# Initialize Model and Convert Data
# =========================================================================================================================================

print("\nInitializing base RoBERTa model and tokenizer...")
model_name = "tner/roberta-large-ontonotes5"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_hf = convert_to_hf_dataset(train_records, tokenizer)
test_hf = convert_to_hf_dataset(test_records, tokenizer)

print("\nSUCCESS: Data Preparation Complete")

# =========================================================================================================================================
# Instantiate Phase 1 Model
# =========================================================================================================================================

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    ignore_mismatched_sizes=True,
    id2label=id_to_label,
    label2id=label_to_id
)

# =========================================================================================================================================
# Train and Save Phase 1 Model
# =========================================================================================================================================

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

training_args = TrainingArguments(
    output_dir="./ner_model_phase1_temp",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=True,
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=test_hf,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Running Phase 1 domain adaptation...")
trainer.train()

# =========================================================================================================================================
# Output Serialization
# =========================================================================================================================================

phase1_model_path = "./ner_model_phase1"
trainer.save_model(phase1_model_path)
tokenizer.save_pretrained(phase1_model_path)

print(f"\nSUCCESS: Phase 1 model saved to {phase1_model_path}")

# =========================================================================================================================================
# Evaluate Phase 1 Model
# =========================================================================================================================================

print("Loading Phase 1 model for evaluation...")
eval_tokenizer = AutoTokenizer.from_pretrained(phase1_model_path)
eval_model = AutoModelForTokenClassification.from_pretrained(phase1_model_path)

eval_nlp_ner = hf_pipeline("ner", model=eval_model, tokenizer=eval_tokenizer, device=0, aggregation_strategy="simple")

def clean_entity_text(text):
    text = re.sub(r'^(the|a|an)\s+', '', text, flags=re.IGNORECASE)
    return text.strip(".,;:!?()[]{}\"' ").lower()

global_metrics = {"tp": 0, "fp": 0, "fn": 0}
cohort_metrics = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
error_count = 0

for record in test_records:
    local_id = str(record.get("local_id") or "Unknown")
    cohort_name = str(record.get("cohort") or "Unknown Cohort").strip()

    target_text = str(record.get("description") or "").strip()

    gt_entities = {(str(ent.get("entity_span") or "").strip().lower(), str(ent.get("label") or "").strip()) for ent in record.get("entities", []) if str(ent.get("label") or "").strip() in TARGET_LABELS}

    if not target_text: continue

    try:
        raw_predictions = eval_nlp_ner(target_text)
        pred_entities = set()

        for pred in raw_predictions:
            label = pred["entity_group"]
            if label in TARGET_LABELS:
                ext = clean_entity_text(target_text[pred["start"]:pred["end"]])
                if len(ext) > 1:
                    pred_entities.add((ext, label))

        matched_gt = set()
        for pred_text, pred_label in pred_entities:
            is_tp = False
            for gt_text, gt_label in gt_entities:
                if pred_label == gt_label and (pred_text in gt_text or gt_text in pred_text):
                    is_tp = True
                    matched_gt.add((gt_text, gt_label))
                    break

            if is_tp:
                global_metrics["tp"] += 1; cohort_metrics[cohort_name]["tp"] += 1
            else:
                global_metrics["fp"] += 1; cohort_metrics[cohort_name]["fp"] += 1

        for gt in gt_entities:
            if gt not in matched_gt:
                global_metrics["fn"] += 1; cohort_metrics[cohort_name]["fn"] += 1

    except Exception as e:
        error_count += 1
        print(f"Error on {local_id}: {e}")

def calc_metrics(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
    return round(p, 4), round(r, 4), round(f1, 4)

cohort_rows = []
for cohort, counts in cohort_metrics.items():
    p, r, f1 = calc_metrics(counts["tp"], counts["fp"], counts["fn"])
    cohort_rows.append({"Cohort": cohort, "TP": counts["tp"], "FP": counts["fp"], "FN": counts["fn"], "Precision": p, "Recall": r, "F1": f1})

gp, gr, gf1 = calc_metrics(global_metrics["tp"], global_metrics["fp"], global_metrics["fn"])
global_df = pd.DataFrame([{"Metric Stage": "Phase 1 Metric Run (Aggregated)", "TP": global_metrics["tp"], "FP": global_metrics["fp"], "FN": global_metrics["fn"], "Precision": gp, "Recall": gr, "F1-Score": gf1}])

print("\nGLOBAL PERFORMANCE")
display(global_df)
print("\nCOHORT BREAKDOWN")
display(pd.DataFrame(cohort_rows))

if error_count > 0:
  print(f"\nSkipped Rows Due to Errors: {error_count}")
else:
  print("\nSUCCESS: Phase 1 Model Evaluated")

### NER Model: Phase 2

In [ ]:
# =========================================================================================================================================
# Re-initialize Model and Convert Data
# =========================================================================================================================================

OUTPUT_FILE = "extractions.json"

print("Initiating Phase 2: 100% Data Production Training...")

prod_model_name = "tner/roberta-large-ontonotes5" # fresh baseline model so it doesn't overfit by training twice
prod_tokenizer = AutoTokenizer.from_pretrained(prod_model_name)
prod_model = AutoModelForTokenClassification.from_pretrained(
    prod_model_name,
    num_labels=len(label_list),
    ignore_mismatched_sizes=True,
    id2label=id_to_label,
    label2id=label_to_id
)

full_hf_dataset = convert_to_hf_dataset(gold_data, prod_tokenizer)

# =========================================================================================================================================
# Train and Save Phase 2 Model
# =========================================================================================================================================

# No eval_strategy or early stopping, as we are training on 100% of the gold data for final deployment
prod_training_args = TrainingArguments(
    output_dir="./ner_model_phase2_temp",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=True,
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False,
    save_strategy="no"
)

prod_trainer = Trainer(
    model=prod_model,
    args=prod_training_args,
    train_dataset=full_hf_dataset, # 100% of the dataset
    data_collator=DataCollatorForTokenClassification(prod_tokenizer),
)

print("\nTraining final production model on all annotated records...")
prod_trainer.train()

# =========================================================================================================================================
# Output Serialization
# =========================================================================================================================================

phase2_model_path = "./ner_model_phase2"
prod_trainer.save_model(phase2_model_path)
prod_tokenizer.save_pretrained(phase2_model_path)

print(f"\nSUCCESS: Phase 2 model saved to {phase2_model_path}")

# =========================================================================================================================================
# Load Data and Phase 2 Model
# =========================================================================================================================================

try:
    with open("rawIngestedMetadata.json", "r", encoding="utf-8") as f:
        ingested_records = json.load(f)
    print(f"Loaded {len(ingested_records)} records for production.")
except FileNotFoundError:
    print("No rawIngestedMetadata.json found. Skipping production.")
    ingested_records = []

results_dict = []

print("\nLoading Phase 2 Production Model...")
prod_tokenizer = AutoTokenizer.from_pretrained(phase2_model_path)
prod_model = AutoModelForTokenClassification.from_pretrained(phase2_model_path)

# =========================================================================================================================================
# Initialize Pipeline
# =========================================================================================================================================

prod_nlp_ner = hf_pipeline(
    "ner",
    model=prod_model,
    tokenizer=prod_tokenizer,
    device=0,
    aggregation_strategy="simple"
)

# =========================================================================================================================================
# Named Entity Extraction Loop
# =========================================================================================================================================

for idx, record in enumerate(ingested_records):
    local_id = record.get("local_id")
    cohort_name = record.get("cohort", "Unknown Cohort")
    theme = record.get("theme", "Unknown Theme")
    title = record.get("title", "Unknown Title")

    target_text = record.get("description", "").strip()

    if not target_text: continue

    document_payload = {
        "local_id": local_id,
        "cohort": cohort_name,
        "theme": theme,
        "title": title,
        "description": target_text,
        "entities": []
    }

    try:
        raw_predictions = prod_nlp_ner(target_text)

        for pred in raw_predictions:
            label = pred["entity_group"]
            score = pred["score"]

            if label in TARGET_LABELS and score >= 0.90:
                ext = clean_entity_text(target_text[pred["start"]:pred["end"]])
                if len(ext) > 1:
                    document_payload["entities"].append({
                        "start": pred["start"],
                        "end": pred["end"],
                        "label": label,
                        "entity_span": ext,
                        "ner_confidence": float(round(score, 4)),
                        "wikidata_id": None # left empty for linking & mapping
                    })

        if document_payload["entities"]:
            seen = set() # deduplicates exact overlapping spans to keep the JSON clean
            unique_entities = []
            for ent in document_payload["entities"]:
                ent_tuple = (ent["start"], ent["end"], ent["label"])
                if ent_tuple not in seen:
                    seen.add(ent_tuple)
                    unique_entities.append(ent)

            document_payload["entities"] = unique_entities
            results_dict.append(document_payload)

    except Exception as e:
        print(f"Error on {local_id}: {e}")

# =========================================================================================================================================
# Output Serialization
# =========================================================================================================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(results_dict, f, indent=4)

print(f"\nSUCCESS: Production extraction saved to '{OUTPUT_FILE}'")

# Linking / Mapping

## Wikidata Candidate Retrieval

In [ ]:
# =========================================================================================================================================
# Configuration
# =========================================================================================================================================

INPUT_FILE = "extractions.json"
OUTPUT_FILE = "candidates.json"

try:
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        records = json.load(f)
    print(f"Loaded {len(records)} records for Wikidata graph-matching candidate search.")
except FileNotFoundError:
    print(f"Error: '{INPUT_FILE}' not found.")
    records = []

def query_wikidata_api(entity_text, limit=5): # this function searches Wikidata usi the API to locate candidates and descriptions
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "search": entity_text,
        "language": "en",
        "format": "json",
        "limit": limit
    }

    headers = {
        "User-Agent": "DSA598NERPipeline/1.0 (grovern@sunypoly.edu)"
    }

    try:
        response = requests.get(url, params=params, headers=headers, timeout=10)
        if response.status_code == 200:
            search_results = response.json().get("search", [])
            candidates = []
            for item in search_results:
                candidates.append({
                    "qid": item.get("id"),
                    "label": item.get("label"),
                    "description": item.get("description", "No description available.")
                })
            return candidates
    except Exception as e:
        print(f"\nNetwork/API Error searching '{entity_text}': {e}")
    return []

# =========================================================================================================================================
# Candidate Retrieval Loop
# =========================================================================================================================================

entity_cache = {} # sets up cache to track already located entities

print("\nStarting Wikidata graph matching candidate retrieval...")
start_time = time.time()

for idx, record in enumerate(records):
    if (idx + 1) % 100 == 0 or (idx + 1) == len(records): # prints progress every 100 records
        print(f"Processing record {idx + 1}/{len(records)}... [Cached {len(entity_cache)} unique entities]")

    for entity in record.get("entities", []): # this loop checks to see if an entity has already been cached first, otherwise will hit the API
        span = entity.get("entity_span")
        if not span:
            continue

        if span in entity_cache:
            entity["wikidata_candidates"] = entity_cache[span]
        else:
            candidates = query_wikidata_api(span, limit=5)
            entity_cache[span] = candidates
            entity["wikidata_candidates"] = candidates

            time.sleep(0.1)

# =========================================================================================================================================
# Output Serialization
# =========================================================================================================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=4)

elapsed_time = round(time.time() - start_time, 2)

print(f"\nSUCCESS: Cached {len(entity_cache)} unique entity spans and saved candidate-augmented dataset to '{OUTPUT_FILE}' in {elapsed_time} seconds")

## LLM Disambiguation

In [ ]:
# =========================================================================================================================================
# Configuration
# =========================================================================================================================================

try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    OPENAI_API_KEY = None

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY NOT FOUND.")

client = OpenAI(api_key=OPENAI_API_KEY)

INPUT_FILE = "candidates_90.json"
OUTPUT_FILE = "linked.json"

CHECKPOINT_INTERVAL = 50 # save progress every 50 records

if os.path.exists(OUTPUT_FILE):
    print(f"Resuming from existing {OUTPUT_FILE}...")
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        records = json.load(f)
else:
    try:
        with open(INPUT_FILE, "r", encoding="utf-8") as f:
            records = json.load(f)
        print(f"Loaded {len(records)} records for Context-Aware LLM Verification.")
    except FileNotFoundError:
        print("No candidate file found.")
        records = []

def verify_with_llm(context_text, entity_span, candidates, peer_entities, cohort, theme):

    try:
        with open("disambiguation_system_prompt.txt", "r", encoding="utf-8") as f:
            system_prompt = f.read().strip()
        with open("disambiguation_user_prompt.txt", "r", encoding="utf-8") as f:
            user_prompt_template = f.read().strip()
    except FileNotFoundError as e:
        print(f"Error loading prompt files: {e}")
        return "NIL", "System error: Prompt text files missing."

    candidate_string = ""
    for idx, c in enumerate(candidates):
        candidate_string += f"[{idx+1}] QID: {c['qid']} | Label: {c['label']} | Description: {c['description']}\n"

    peer_string = ", ".join(peer_entities) if peer_entities else "None"

    user_prompt = user_prompt_template.format(
        cohort=cohort,
        theme=theme,
        context_text=context_text,
        peer_entities=peer_string,
        target_entity=entity_span,
        candidates=candidate_string
    )

    max_retries = 6
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-5.4-mini",
                temperature=0.0,
                max_completion_tokens=150,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ]
            )

            output_json = json.loads(response.choices[0].message.content.strip())
            decision = output_json.get("wikidata_id", "NIL").strip().upper()
            reasoning = output_json.get("reasoning", "No explicit reasoning provided by model.")
            return decision, reasoning

        except RateLimitError:
            sleep_time = 2 ** attempt
            time.sleep(sleep_time)
        except json.JSONDecodeError:
            return "NIL", "Failed to parse valid JSON response from LLM."
        except Exception as e:
            return "NIL", f"API Error encountered: {e}"

    return "NIL", "API retries exhausted."

# =========================================================================================================================================
# LLM Disambiguation Loop
# =========================================================================================================================================

print("\nStarting Context-Aware LLM Verification...")
entities_linked_count = 0

for idx, record in enumerate(tqdm(records, desc="Processing Records", unit="rec")):
    title_context = record.get("title", "Unknown Title")
    desc_context = record.get("description", "")
    cohort = record.get("cohort", "Unknown Cohort")
    theme = record.get("theme", "Unknown Theme")

    text_context = f"TITLE: {title_context}\nDESCRIPTION: {desc_context}"  # combine title and description to give the LLM maximum context for disambiguation

    all_spans_in_doc = [e.get("entity_span", "") for e in record.get("entities", []) if e.get("entity_span")]

    modified_in_this_record = False
    for entity in record.get("entities", []):

        if entity.get("wikidata_id") is not None and entity.get("wikidata_id") != "": # SKIP if already processed
            continue

        candidates = entity.get("wikidata_candidates", [])
        span = entity.get("entity_span", "")

        if len(candidates) == 0:
            decision = "NIL"
            reasoning = "Skipped LLM step: No candidates generated by the Wikidata API."
        else:
            peer_entities = [p for p in all_spans_in_doc if p != span]
            decision, reasoning = verify_with_llm(
                text_context,
                span,
                candidates,
                peer_entities,
                cohort,
                theme
            )

            if not (decision.startswith("Q") or decision == "NIL"):
                decision = "NIL"
                reasoning += " [Fallback enforced: Invalid ID structure returned.]"

        entity["wikidata_id"] = decision
        entity["llm_reasoning"] = reasoning
        entities_linked_count += 1
        modified_in_this_record = True
        time.sleep(0.05)

    if (idx + 1) % CHECKPOINT_INTERVAL == 0 and modified_in_this_record:  # save progress periodically
        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            json.dump(records, f, indent=4)

# =========================================================================================================================================
# Output Serialization
# =========================================================================================================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=4)

print(f"\nSUCCESS: Finished processing. Total new entities linked: {entities_linked_count}. Saved pipeline data with audit trails to '{OUTPUT_FILE}'")


## Linking / Mapping Evaluation

### Hallucination Evaluation

In [ ]:
# =========================================================================================================================================
# Configuration
# =========================================================================================================================================

JUDGE_MODEL = "gpt-4o"
SAMPLE_SIZE = 99

try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    OPENAI_API_KEY = None

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY NOT FOUND.")

client = OpenAI(api_key=OPENAI_API_KEY)

INPUT_FILE = "linked.json"

try:
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        records = json.load(f)
    print(f"Loaded {len(records)} records for advanced pipeline diagnostics.")
except FileNotFoundError:
    print(f"Error: '{INPUT_FILE}' not found.")
    records = []

# defining measures
total_audits = 0
format_errors = 0
boundary_errors = 0
reasoning_errors = 0

qid_pattern = re.compile(r"^Q\d+$")
diagnostic_audit_trail = []

# =========================================================================================================================================
# Linking Evaluation Function
# =========================================================================================================================================

def audit_linking_decision(cohort, theme, context, entity_span, entity_label, decision, reasoning, candidates):

    # Evaluates both selection accuracy and potential NIL under-linking cowardice using
    # external prompts, classifying errors into structured typologies.

    try:
        with open("hallucination_system_prompt.txt", "r", encoding="utf-8") as f:
            system_prompt = f.read().strip()
        with open("hallucination_user_prompt.txt", "r", encoding="utf-8") as f:
            user_prompt_template = f.read().strip()
    except FileNotFoundError as e:
        return True, "None", f"Audit failed: Prompt text files missing. {e}"

    # format candidates list for the judge
    candidates_list = ""
    for idx, c in enumerate(candidates):
        candidates_list += f"[{idx+1}] QID: {c['qid']} | Label: {c['label']} | Description: {c['description']}\n"

    # format user prompt
    user_prompt = user_prompt_template.format(
        cohort=cohort,
        theme=theme,
        context=context,
        entity_span=entity_span,
        entity_label=entity_label,
        decision=decision,
        reasoning=reasoning,
        candidates_list=candidates_list if candidates_list else "No candidates available."
    )

    try:
        response = client.chat.completions.create(
            model=JUDGE_MODEL,
            temperature=0.0,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )
        output = json.loads(response.choices[0].message.content.strip())
        return (
            output.get("is_error", False),
            output.get("error_typology", "None"),
            output.get("audit_reasoning", "No details provided.")
        )
    except Exception as e:
        return False, "None", f"Audit connection skipped: {e}"

# =========================================================================================================================================
# Stratified Cohort Sampling
# =========================================================================================================================================

records_by_cohort = {}
for r in records:
    cohort = r.get("cohort", "Unknown Cohort")
    records_by_cohort.setdefault(cohort, []).append(r)

sampled_records = []
if records:
    # distribute sample size evenly across cohorts
    cohorts = list(records_by_cohort.keys())
    samples_per_cohort = max(1, SAMPLE_SIZE // len(cohorts))

    for cohort, cohort_recs in records_by_cohort.items():
        sample_pool = random.sample(cohort_recs, min(len(cohort_recs), samples_per_cohort))
        sampled_records.extend(sample_pool)

print(f"\nStratified sampling complete: Auditing {len(sampled_records)} records out of {len(records)} total records.")

# =========================================================================================================================================
# Execution Loop
# =========================================================================================================================================

print(f"\nEvaluating pipeline using Judge: '{JUDGE_MODEL}'...")

for record in sampled_records:
    # pull cohort and theme metadata
    cohort = record.get("cohort", "Unknown Cohort")
    theme = record.get("theme", "Unknown Theme")

    title_context = record.get("title", "Unknown Title")
    desc_context = record.get("description", "")
    text_context = f"TITLE: {title_context}\nDESCRIPTION: {desc_context}"

    for entity in record.get("entities", []):
        candidates = entity.get("wikidata_candidates", [])

        if not candidates:
            continue

        total_audits += 1
        decision = str(entity.get("wikidata_id", "")).strip()
        reasoning = str(entity.get("llm_reasoning", "")).strip()
        entity_span = str(entity.get("entity_span", "")).strip()
        entity_label = str(entity.get("label", "Unknown")).strip()

        # check structure
        if decision != "NIL" and not qid_pattern.match(decision):
            format_errors += 1
            diagnostic_audit_trail.append({
                "cohort": cohort,
                "entity_type": entity_label,
                "entity_span": entity_span,
                "decision": decision,
                "reasoning_type": "Structural",
                "error_typology": "Format Error",
                "audit_reasoning": "Model returned an ID that does not match standard QID format regex."
            })
            continue

        # check boundaries
        if decision != "NIL":
            valid_candidate_qids = [c.get("qid") for c in candidates]
            if decision not in valid_candidate_qids:
                boundary_errors += 1
                diagnostic_audit_trail.append({
                    "cohort": cohort,
                    "entity_type": entity_label,
                    "entity_span": entity_span,
                    "decision": decision,
                    "reasoning_type": "Structural",
                    "error_typology": "Boundary Violation",
                    "audit_reasoning": f"Model selected QID '{decision}', which was not in the verified candidate list."
                })
                continue

        # check semantics & NIL cowardice
        is_error, typology, audit_reason = audit_linking_decision(
            cohort=cohort,
            theme=theme,
            context=text_context,
            entity_span=entity_span,
            entity_label=entity_label,
            decision=decision,
            reasoning=reasoning,
            candidates=candidates
        )

        if is_error:
            reasoning_errors += 1
            diagnostic_audit_trail.append({
                "cohort": cohort,
                "entity_type": entity_label,
                "entity_span": entity_span,
                "decision": decision,
                "reasoning_type": "Reasoning",
                "error_typology": typology,
                "audit_reasoning": audit_reason
            })

        time.sleep(0.05)

# =========================================================================================================================================
# Strategic Aggregation & Data Reporting
# =========================================================================================================================================

print("\nLinking Evaluation")

# global metrics
global_err_rate = (len(diagnostic_audit_trail) / total_audits) if total_audits > 0 else 0
summary_df = pd.DataFrame([{
    "Total Linkings Audited": total_audits,
    "Format Errors": format_errors,
    "Boundary Violations": boundary_errors,
    "Reasoning Errors": reasoning_errors,
    "Total Pipeline Errors": len(diagnostic_audit_trail),
    "Global Error Rate": f"{global_err_rate:.2%}"
}])
print("\n[GLOBAL METRIC SUMMARY]")
display(summary_df)

# check for errors
if diagnostic_audit_trail:
    err_df = pd.DataFrame(diagnostic_audit_trail)

    # breakdown typology error
    typology_counts = err_df["error_typology"].value_counts().reset_index()
    typology_counts.columns = ["Error Typology Type", "Count"]
    print("\n[ERROR TYPOLOGY DECOMPOSITION]")
    display(typology_counts)

    # error analysis by cohort
    cohort_stats = err_df.groupby("cohort").size().reset_index(name="Errors Found")
    print("\n[ERRORS SEGMENTED BY ARCHIVAL COHORT]")
    display(cohort_stats)

    # entity type analysis
    label_stats = err_df.groupby("entity_type").size().reset_index(name="Errors Found")
    print("\n[VULNERABILITY BY ENTITY TYPE (NER CLASS)]")
    display(label_stats)

    # raw error table
    print("\n[RAW QUALITY AUDIT SAMPLES]")
    pd.set_option('display.max_colwidth', None)
    display(err_df[["cohort", "entity_span", "decision", "error_typology", "audit_reasoning"]].head(10))

else:
    print("\nSUCCESS: 0 errors detected across all formats, boundaries, and historical reasonings!")

### Quantitative Evaluation

In [ ]:
# =========================================================================================================================================
# Configuration
# =========================================================================================================================================

GOLD_FILE = "manualAnnotationFinal.json"
PREDICTED_FILE = "linked.json"

try:
    with open(GOLD_FILE, "r", encoding="utf-8") as f:
        gold_records = json.load(f)
    with open(PREDICTED_FILE, "r", encoding="utf-8") as f:
        predicted_records = json.load(f)
    print(f"Loaded {len(gold_records)} gold standard records and {len(predicted_records)} system predicted records.")
except FileNotFoundError as e:
    print(f"Error loading files: {e}. Please ensure both files are in the working directory.")
    gold_records, predicted_records = [], []

# map predictions by local_id for rapid lookups
pred_map = {r.get("local_id"): r for r in predicted_records if r.get("local_id")}

# template dictionary for tracking structured metrics
def create_metric_bucket():
    return {
        "candidate_hits": 0, # correct QID was in candidate list
        "gold_inkb_count": 0, # total entities that actually exist in Wikidata (non-NIL)
        "inkb_tp": 0, "inkb_fp": 0, "inkb_fn": 0,
        "nil_tp": 0, "nil_fp": 0, "nil_fn": 0
    }

global_metrics = create_metric_bucket()
cohort_metrics = {} # cohort segmentation

# =========================================================================================================================================
# Quantitative Calculation Loop
# =========================================================================================================================================

for gold_record in gold_records:
    local_id = gold_record.get("local_id")
    cohort = gold_record.get("cohort", "Unknown Cohort")

    if cohort not in cohort_metrics:
        cohort_metrics[cohort] = create_metric_bucket()

    pred_record = pred_map.get(local_id)
    if not pred_record:
        continue # skip if document was missing from prediction

    # map predicted entities by offset boundaries
    pred_entities = {
        (e.get("start"), e.get("end")): e
        for e in pred_record.get("entities", [])
    }

    for gold_ent in gold_record.get("entities", []):
        start = gold_ent.get("start")
        end = gold_ent.get("end")
        gold_qid = str(gold_ent.get("wikidata_id", "NIL")).strip().upper()

        # match predicted entity using character spans
        pred_ent = pred_entities.get((start, end))
        if not pred_ent:
            continue # only evaluate linking on spans successfully extracted by NER

        pred_qid = str(pred_ent.get("wikidata_id", "NIL")).strip().upper()
        candidates = pred_ent.get("wikidata_candidates", [])
        candidate_qids = [str(c.get("qid")).strip().upper() for c in candidates]

        # track candidate generation recall
        if gold_qid != "NIL":
            global_metrics["gold_inkb_count"] += 1
            cohort_metrics[cohort]["gold_inkb_count"] += 1
            if gold_qid in candidate_qids:
                global_metrics["candidate_hits"] += 1
                cohort_metrics[cohort]["candidate_hits"] += 1

        # track in knowledge-base vs NIL precisiona nd recall
        if gold_qid != "NIL":  # entity EXISTS in ground truth
            if pred_qid == gold_qid:
                global_metrics["inkb_tp"] += 1
                cohort_metrics[cohort]["inkb_tp"] += 1
            elif pred_qid == "NIL":
                global_metrics["inkb_fn"] += 1
                cohort_metrics[cohort]["inkb_fn"] += 1
            else:
                global_metrics["inkb_fp"] += 1
                cohort_metrics[cohort]["inkb_fp"] += 1
        else:  # entity DOES NOT EXIST in ground truth (NIL)
            if pred_qid == "NIL":
                global_metrics["nil_tp"] += 1
                cohort_metrics[cohort]["nil_tp"] += 1
            else:
                global_metrics["nil_fp"] += 1
                global_metrics["nil_fn"] += 1
                cohort_metrics[cohort]["nil_fp"] += 1
                cohort_metrics[cohort]["nil_fn"] += 1

# =========================================================================================================================================
# Statistics Processing Functions
# =========================================================================================================================================

def calculate_precision_recall_f1(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def compile_metrics_row(label, metrics_dict):
    cand_recall = metrics_dict["candidate_hits"] / metrics_dict["gold_inkb_count"] if metrics_dict["gold_inkb_count"] > 0 else 0.0

    inkb_prec, inkb_rec, inkb_f1 = calculate_precision_recall_f1(
        metrics_dict["inkb_tp"], metrics_dict["inkb_fp"], metrics_dict["inkb_fn"]
    )

    nil_prec, nil_rec, nil_f1 = calculate_precision_recall_f1(
        metrics_dict["nil_tp"], metrics_dict["nil_fp"], metrics_dict["nil_fn"]
    )

    return {
        "Scope / Cohort": label,
        "Candidate Recall@5": f"{cand_recall:.2%}",
        "In-KB Precision": f"{inkb_prec:.2%}",
        "In-KB Recall": f"{inkb_rec:.2%}",
        "In-KB F1": f"{inkb_f1:.2%}",
        "NIL Precision": f"{nil_prec:.2%}",
        "NIL Recall": f"{nil_rec:.2%}",
        "NIL F1": f"{nil_f1:.2%}"
    }

# =========================================================================================================================================
# Results
# =========================================================================================================================================

# generate global baseline row
global_row = compile_metrics_row("GLOBAL BASELINE (All Cohorts)", global_metrics)
global_df = pd.DataFrame([global_row])

# generate per-cohort rows
cohort_rows = []
for cohort, metrics in cohort_metrics.items():
    cohort_rows.append(compile_metrics_row(cohort, metrics))
cohort_df = pd.DataFrame(cohort_rows)

print("\n" + "="*110)
print("QUANTITATIVE ENTITY LINKING DIAGNOSTIC REPORT (COHORT SEGMENTED)")
print("="*110)

print("\n[GLOBAL RESULTS]")
display(global_df)

print("\n[RESULTS SEGMENTED BY ARCHIVAL COHORT]")
display(cohort_df.sort_values(by="In-KB F1", ascending=False))

# Semantic Enrichment / Visualization

## NIL Clustering

In [ ]:
# =========================================================================================================================================
# Configuration
# =========================================================================================================================================

INPUT_FILE = "linked.json"
OUTPUT_FILE = "clustered.json"

FUZZY_THRESHOLD = 90  # strict matching boundary
ISOLATE_COHORTS = True # if True, prevents matching the same NIL string across different cohorts

# similarity wrapper
try:
    from rapidfuzz import fuzz
    SIMILARITY_ENGINE = "rapidfuzz"
except ImportError:
    SIMILARITY_ENGINE = "difflib"

def calculate_similarity(s1, s2):

    # Computes string similarity score [0, 100] using the best available engine.

    if SIMILARITY_ENGINE in ["rapidfuzz", "fuzzywuzzy", "thefuzz"]:
        return fuzz.ratio(s1, s2)
    else:
        import difflib
        return difflib.SequenceMatcher(None, s1, s2).ratio() * 100

# =========================================================================================================================================
# Structural Key Processing
# =========================================================================================================================================

def normalize_span(span_text):
    if not span_text: return ""
    return span_text.strip().lower()

def get_clean_blocking_key(span_text):

    # Strips leading historical honorifics and articles
    # to prevent blocking misalignment (e.g. "The Cherokee Nation" -> "c").

    if not span_text:
        return "?"
    text = span_text.strip().lower()

    prefixes = [
        "the ", "a ", "an ", "dr. ", "dr ", "mr. ", "mr ", "mrs. ", "mrs ", "ms. ", "ms ", "rev. ", "rev ", "hon. ", "hon "
    ]
    cleaned = text
    for prefix in prefixes:
        if cleaned.startswith(prefix):
            cleaned = cleaned[len(prefix):].strip()
            break # only strip the outer-most matched prefix

    if cleaned and cleaned[0].isalnum():
        return cleaned[0]
    return text[0] if text else "?"

def get_compound_block_key(norm_span, label, cohort):

    # Enforces cohort boundaries and entity label types
    # to prevent unrelated entity types and historical eras from merging.

    clean_char = get_clean_blocking_key(norm_span)
    if ISOLATE_COHORTS:
        return (clean_char, label, cohort)
    return (clean_char, label)

# =========================================================================================================================================
# Registry Access Blocks
# =========================================================================================================================================

def find_fuzzy_match_blocked(new_span, block_key, registry, threshold=90):
    if not new_span: return None

    # extract only matching characters, type, and cohort block
    target_block = registry.get(block_key, {})

    for existing_span, nil_id in target_block.items():
        if calculate_similarity(new_span, existing_span) >= threshold:
            return nil_id
    return None

def register_nil_entity(new_span, nil_id, block_key, registry):
    if not new_span: return
    if block_key not in registry:
        registry[block_key] = {}
    registry[block_key][new_span] = nil_id

# =========================================================================================================================================
# Execution Loop
# =========================================================================================================================================

try:
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        records = json.load(f)
    print(f"Loaded {len(records)} records. Matching strings using '{SIMILARITY_ENGINE}' engine...")
except FileNotFoundError:
    print(f"Error: '{INPUT_FILE}' not found.")
    records = []

nil_registry = {}  # format: { (clean_char, label, cohort): { norm_span: nil_id } }
nil_counter = 1
total_propagations = 0

print("Starting Canopy-Blocked Co-reference Propagation...")

for idx, record in enumerate(records):
    cohort = record.get("cohort", "Unknown Cohort")
    entities = record.get("entities", [])
    if not entities: continue

    sorted_entities = sorted(entities, key=lambda x: len(x.get("entity_span", "")), reverse=True)

    # local anchor isolation
    document_anchors = {}
    for ent in sorted_entities:
        span = normalize_span(ent.get("entity_span", ""))
        label = ent.get("label", "")
        if label in ["PERSON", "ORG", "GPE"] and span:
            if not any(span != long_span and span in long_span for long_span in document_anchors.keys()):
                document_anchors[span] = ent

    # identify inner-document substring coreferences
    for ent in sorted_entities:
        # skip entities that already have a valid Wikidata QID
        if str(ent.get("wikidata_id", "")).startswith("Q"):
            continue

        norm_span = normalize_span(ent.get("entity_span", ""))
        label = ent.get("label", "")
        if not norm_span or label not in ["PERSON", "ORG", "GPE"]:
            continue

        matching_parents = [long_span for long_span in document_anchors.keys() if norm_span != long_span and norm_span in long_span]
        if len(matching_parents) == 1:
            ent["_coref_parent_span"] = matching_parents[0]
            total_propagations += 1

    # core rntity NIL clustering (skips local child coreferences)
    for ent in sorted_entities:
        if "_coref_parent_span" in ent:
            continue

        norm_span = normalize_span(ent.get("entity_span", ""))
        if not norm_span: continue

        if ent.get("wikidata_id") == "NIL":
            label = ent.get("label", "UNKNOWN")
            block_key = get_compound_block_key(norm_span, label, cohort)

            # look for absolute exact matches in this specific compound block
            if block_key in nil_registry and norm_span in nil_registry[block_key]:
                ent["wikidata_id"] = nil_registry[block_key][norm_span]
            else:
                # check for fuzzy matches within this specific compound block
                fuzzy_id = find_fuzzy_match_blocked(norm_span, block_key, nil_registry, threshold=FUZZY_THRESHOLD)
                if fuzzy_id:
                    ent["wikidata_id"] = fuzzy_id
                    register_nil_entity(norm_span, fuzzy_id, block_key, nil_registry)
                else:
                    # mint a fresh global ID
                    new_id = f"NIL-{nil_counter:04d}"
                    register_nil_entity(norm_span, new_id, block_key, nil_registry)
                    ent["wikidata_id"] = new_id
                    nil_counter += 1

    # propagate parent IDs to child coreferences inside same document
    for ent in sorted_entities:
        if "_coref_parent_span" in ent:
            parent_span = ent["_coref_parent_span"]
            parent_entity = document_anchors[parent_span]
            ent["wikidata_id"] = parent_entity.get("wikidata_id")
            del ent["_coref_parent_span"]

# =========================================================================================================================================
# Output Serialization
# =========================================================================================================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=4)

# calculate statistics
total_nil_mentions = 0
all_assigned_nils = []

for r in records:
    for e in r.get("entities", []):
        w_id = e.get("wikidata_id", "")
        if str(w_id).startswith("NIL-"):
            total_nil_mentions += 1
            all_assigned_nils.append((w_id, e.get("entity_span", ""), e.get("label", ""), r.get("cohort", "")))

unique_nil_clusters = len(set(x[0] for x in all_assigned_nils))
compression_ratio = (total_nil_mentions / unique_nil_clusters) if unique_nil_clusters > 0 else 1.0

# extract top clustered items for validation
cluster_distributions = Counter(x[0] for x in all_assigned_nils)
top_5_clusters = cluster_distributions.most_common(5)

top_clusters_report = []
for cluster_id, count in top_5_clusters:
    associated_mentions = list(set([x[1] for x in all_assigned_nils if x[0] == cluster_id]))
    sample_labels = list(set([x[2] for x in all_assigned_nils if x[0] == cluster_id]))
    sample_cohorts = list(set([x[3] for x in all_assigned_nils if x[0] == cluster_id]))

    top_clusters_report.append({
        "Cluster ID": cluster_id,
        "Total Mentions": count,
        "Unique Surface Spans": ", ".join(associated_mentions[:4]),
        "Entity Labels": ", ".join(sample_labels),
        "Cohort Groupings": ", ".join(sample_cohorts)
    })

top_clusters_df = pd.DataFrame(top_clusters_report)

print("\nGRAPH COMPRESSION & CLUSTERING DIAGNOSTICS")
print(f"\n• Propagated {total_propagations} inner-document name splits (e.g. 'Smith' -> 'John Smith').")
print(f"• Resolved {total_nil_mentions} raw NIL entity mentions into {unique_nil_clusters} distinct clusters.")
print(f"• Graph Compression Ratio: {compression_ratio:.3f}x (Mentions consolidated per ID).")
print("\n[TOP 5 LARGEST NIL CLUSTERS (HUMAN VERIFICATION AUDIT)]")
display(top_clusters_df)
print(f"\nSUCCESS: Saved clean clustered dataset to '{OUTPUT_FILE}'")

## Semantic Density Enrichment (JSON-LD)

In [ ]:
# =========================================================================================================================================
# Configuration
# =========================================================================================================================================

INPUT_FILE = "clustered.json"
OUTPUT_FILE = "enriched.jsonld"

# structural properties mapping the 5 entity labels
TARGET_PROPERTIES = {
    "P31": "instanceOf",
    "P131": "locatedIn",
    "P106": "occupation",

    # merging citizenship and country into the same demographic bucket for the dashboard
    "P27": "country",
    "P17": "country",

    # temporal properties (lifespans)
    "P569": "dateOfBirth",
    "P570": "dateOfDeath",

    # temporal properties (events & organizations)
    "P580": "startDate",
    "P582": "endDate",
    "P571": "startDate",
    "P576": "endDate",

    # specific contextual properties
    "P463": "memberOf",
    "P1399": "convictedOf",
    "P21": "genderIdentity",
    "P172": "ethnicGroup",
    "P140": "religion",
    "P1142": "politicalIdeology",
    "P585": "pointInTime",
    "P710": "participant",

    # visualization Properties
    "P18": "image",
    "P625": "geo"
}

# strict class mapping
NER_TO_SCHEMA_MAP = {
    "PERSON": "schema:Person",
    "ORG": "schema:Organization",
    "GPE": "schema:Place",
    "EVENT": "schema:Event",
    "NORP": ["schema:Organization", "local:NORP"] # multi-typed for demographic groups
}

# downstream styling conventions
VISUAL_CONFIG = {
    "schema:Person": {"group": "Person", "color": "#FF5733", "icon": "person"},
    "schema:Organization": {"group": "Organization", "color": "#2ECC71", "icon": "group"},
    "local:NORP": {"group": "Demographic (NORP)", "color": "#9B59B6", "icon": "demographics"},
    "schema:Place": {"group": "Location", "color": "#3498DB", "icon": "place"},
    "schema:Event": {"group": "Historical Event", "color": "#F1C40F", "icon": "event"},
    "schema:Thing": {"group": "Other", "color": "#95A5A6", "icon": "help"}
}

def get_visual_styling(resolved_types):
    # Supplies direct hexadecimal colors, groups, and icons to entities
    # so visualization software can auto-render clusters without custom CSS.
    primary_type = resolved_types[-1] if isinstance(resolved_types, list) else resolved_types
    style = VISUAL_CONFIG.get(primary_type, VISUAL_CONFIG["schema:Thing"])
    return {
        "local:visualGroup": style["group"],
        "local:visualColor": style["color"],
        "local:visualIcon": style["icon"]
    }

# global cache to hold API properties
qid_cache = {}
secondary_qids = set()

# =========================================================================================================================================
# File Loading
# =========================================================================================================================================

try:
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        records = json.load(f)
except FileNotFoundError:
    print(f"Error: Could not find {INPUT_FILE}. Please ensure the file exists.")
    records = []

# =========================================================================================================================================
# Wikidata Batch Collector
# =========================================================================================================================================

def fetch_wikidata_batch_rich(qids, session):
    # Downloads claims, English labels, and descriptions in chunks of 50 in a single network pass.
    if not qids:
        return {}

    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbgetentities",
        "ids": "|".join(qids),
        "props": "claims|labels|descriptions",
        "languages": "en",
        "format": "json"
    }
    headers = {
        "User-Agent": "DSA598CapstoneProjectSemanticEnrichment/1.0 (grovern@sunypoly.edu)"
    }

    max_retries = 3
    backoff = 1.0

    for attempt in range(max_retries):
        try:
            response = session.get(url, params=params, headers=headers, timeout=15)
            if response.status_code == 200:
                data = response.json()
                return data.get("entities", {})
            elif response.status_code == 429:
                time.sleep(backoff)
                backoff *= 2
        except requests.exceptions.RequestException:
            time.sleep(backoff)
            backoff *= 2

    return {}

def process_rich_batch(entities_data):
    # Extracts relations, coordinates, images, and tooltips from API response.
    for qid, entity_info in entities_data.items():
        if "claims" not in entity_info:
            qid_cache[qid] = None
            continue

        claims = entity_info.get("claims", {})
        labels = entity_info.get("labels", {})
        descriptions = entity_info.get("descriptions", {})

        enrichment = {
            "official_label": labels.get("en", {}).get("value"),
            "description": descriptions.get("en", {}).get("value"),
            "viaf_id": None,
            "paths": {}
        }

        # extract VIAF ID
        if "P214" in claims:
            try:
                enrichment["viaf_id"] = claims["P214"][0]["mainsnak"]["datavalue"]["value"]
            except (KeyError, IndexError):
                pass

        # extract structural, temporal, identity, and visual claims
        for prop_id, prop_name in TARGET_PROPERTIES.items():
            if prop_id in claims:
                nodes = []
                for claim in claims[prop_id][:2]: # strict boundary of 2 links to avoid graph clutter
                    try:
                        mainsnak = claim.get("mainsnak", {})
                        if "datavalue" in mainsnak:
                            data_type = mainsnak["datavalue"]["type"]

                            # standard object entities (schema:Thing)
                            if data_type == "wikibase-entityid":
                                connected_qid = mainsnak["datavalue"]["value"]["id"]
                                nodes.append(f"wd:{connected_qid}")
                                secondary_qids.add(connected_qid) # Collect for label resolution later

                            # dates / temporal boundaries (literal string)
                            elif data_type == "time":
                                raw_time = mainsnak["datavalue"]["value"]["time"]
                                clean_time = raw_time.lstrip('+').split('T')[0]
                                nodes.append(clean_time)

                            # images (Parsed dynamically into direct URLs)
                            elif data_type == "string" and prop_id == "P18":
                                filename = mainsnak["datavalue"]["value"]
                                img_url = f"https://commons.wikimedia.org/wiki/Special:FilePath/{quote(filename)}"
                                nodes.append(img_url)

                            # geolocation Coordinates
                            elif data_type == "globecoordinate" and prop_id == "P625":
                                geo_val = mainsnak["datavalue"]["value"]
                                nodes.append({
                                    "@type": "schema:GeoCoordinates",
                                    "latitude": geo_val.get("latitude"),
                                    "longitude": geo_val.get("longitude")
                                })
                    except (KeyError, IndexError):
                        continue

                if nodes:
                    # check if the property already exists in the dict (e.g. combined inception and start dates)
                    if prop_name in enrichment["paths"]:
                        existing = enrichment["paths"][prop_name]
                        if not isinstance(existing, list):
                            existing = [existing]
                        enrichment["paths"][prop_name] = existing + nodes
                    else:
                        enrichment["paths"][prop_name] = nodes[0] if len(nodes) == 1 else nodes

        qid_cache[qid] = enrichment

# =========================================================================================================================================
# Execution & JSON-LD Construction
# =========================================================================================================================================

if not records:
    print("Exiting: No records to process.")
    exit()

# aggregate unique entities
all_unique_qids = set()
for record in records:
    for entity in record.get("entities", []):
        w_id = entity.get("wikidata_id", "NIL")
        if w_id.startswith("Q"):
            all_unique_qids.add(w_id)

print(f"Aggregated {len(all_unique_qids)} unique Wikidata QIDs to enrich.")

# retrieve batch metadata
qids_list = list(all_unique_qids)
batch_size = 50

print(f"Extracting properties and visualization coordinates (Chunk Size: {batch_size})...")
with requests.Session() as session:
    for i in range(0, len(qids_list), batch_size):
        chunk = qids_list[i : i + batch_size]
        entities_data = fetch_wikidata_batch_rich(chunk, session)
        process_rich_batch(entities_data)

        completed = min(i + batch_size, len(qids_list))
        print(f" -> Enriched {completed}/{len(qids_list)} items...")
        time.sleep(0.2)

print(f"\nResolving English labels for {len(secondary_qids)} secondary graph properties...")
secondary_labels = {}
sec_qids_list = list(secondary_qids)

with requests.Session() as session:
    for i in range(0, len(sec_qids_list), batch_size):
        chunk = sec_qids_list[i : i + batch_size]
        entities_data = fetch_wikidata_batch_rich(chunk, session)
        for qid, entity_info in entities_data.items():
            label = entity_info.get("labels", {}).get("en", {}).get("value", qid)
            secondary_labels[qid] = label

        completed = min(i + batch_size, len(sec_qids_list))
        print(f" -> Resolved {completed}/{len(sec_qids_list)} secondary labels...")
        time.sleep(0.2)

# model JSON-LD context and graph nodes
json_ld_document = {
    "@context": {
        "schema": "http://schema.org/",
        "wd": "http://www.wikidata.org/entity/",
        "wdt": "http://www.wikidata.org/prop/direct/",
        "viaf": "http://viaf.org/viaf/",
        "local": "http://your-archive.org/ontology/",
        "Record": "schema:ArchiveComponent",
        "text": "schema:text",
        "cohort": "schema:collection",
        "entities": "schema:mentions",
        "entity_span": "schema:alternateName",
        "ner_confidence": "local:nerConfidence",

        # base entity
        "officialName": "schema:name",
        "description": "schema:description",
        "image": {"@id": "schema:image", "@type": "@id"},

        # visual styling
        "visualColor": "local:visualColor",
        "visualGroup": "local:visualGroup",
        "visualIcon": "local:visualIcon",

        # spatial context
        "geo": "schema:geo",
        "latitude": "schema:latitude",
        "longitude": "schema:longitude",

        # structural graph predicates
        "instanceOf": {"@id": "wdt:P31", "@type": "@id"},
        "locatedIn": {"@id": "wdt:P131", "@type": "@id"},
        "occupation": {"@id": "wdt:P106", "@type": "@id"},

        # temporal bounds
        "startDate": {"@id": "schema:startDate"},
        "endDate": {"@id": "schema:endDate"},
        "dateOfBirth": {"@id": "wdt:P569"},
        "dateOfDeath": {"@id": "wdt:P570"},

        #combined demographics
        "memberOf": {"@id": "wdt:P463", "@type": "@id"},
        "convictedOf": {"@id": "wdt:P1399", "@type": "@id"},
        "genderIdentity": {"@id": "wdt:P21", "@type": "@id"},
        "ethnicGroup": {"@id": "wdt:P172", "@type": "@id"},
        "religion": {"@id": "wdt:P140", "@type": "@id"},
        "politicalIdeology": {"@id": "wdt:P1142", "@type": "@id"},
        "pointInTime": {"@id": "wdt:P585"},
        "participant": {"@id": "wdt:P710", "@type": "@id"},
        "country": {"@id": "wdt:P17", "@type": "@id"}
    },
    "@graph": []
}

total_paths_added = 0
total_images_mapped = 0
total_coordinates_mapped = 0
total_viaf_links = 0

for idx, record in enumerate(records):
    semantic_record = {
        "@type": "Record",
        "@id": f"local:record/{record.get('local_id', idx)}",
        "cohort": record.get("cohort", "Unknown"),
        "text": record.get("text", ""),
        "entities": []
    }

    for ent_idx, entity in enumerate(record.get("entities", [])):
        linked_id = entity.get("wikidata_id", "NIL")
        ner_label = entity.get("label", "UNKNOWN")

        resolved_type = NER_TO_SCHEMA_MAP.get(ner_label, "schema:Thing")
        visuals = get_visual_styling(resolved_type)

        semantic_entity = {
            "@type": resolved_type,
            "entity_span": entity.get("entity_span", ""),
            "ner_confidence": entity.get("ner_confidence", entity.get("confidence", 1.0)),
            "visualColor": visuals["local:visualColor"],
            "visualGroup": visuals["local:visualGroup"],
            "visualIcon": visuals["local:visualIcon"]
        }

        # Case A: Linked to Wikidata (Enrich node with fetched metadata)
        if linked_id.startswith("Q"):
            semantic_entity["@id"] = f"wd:{linked_id}"
            enrichment_data = qid_cache.get(linked_id)

            if enrichment_data:
                # add tooltips
                if enrichment_data["official_label"]:
                    semantic_entity["officialName"] = enrichment_data["official_label"]
                if enrichment_data["description"]:
                    semantic_entity["description"] = enrichment_data["description"]

                # link VIAF Identifiers
                if enrichment_data["viaf_id"]:
                    semantic_entity["schema:sameAs"] = f"viaf:{enrichment_data['viaf_id']}"
                    total_viaf_links += 1

                # unpack Graph paths, Images, and GIS Coordinates
                for path_name, path_value in enrichment_data["paths"].items():
                    if path_name in ["image", "geo"]:
                        semantic_entity[path_name] = path_value
                        if path_name == "image":
                            total_images_mapped += 1
                        elif path_name == "geo":
                            total_coordinates_mapped += 1
                    else:
                        # resolve secondary nodes into rich semantic objects
                        resolved_nodes = []
                        val_list = path_value if isinstance(path_value, list) else [path_value]

                        for val in val_list:
                            if isinstance(val, str) and val.startswith("wd:"):
                                qid = val.replace("wd:", "")
                                label = secondary_labels.get(qid, qid)
                                resolved_nodes.append({"@id": f"wd:{qid}", "schema:name": label})
                            else:
                                resolved_nodes.append(val)

                        semantic_entity[path_name] = resolved_nodes[0] if len(resolved_nodes) == 1 else resolved_nodes
                        total_paths_added += len(resolved_nodes)

        # Case B: Local Clustered NIL Entity
        elif linked_id.startswith("NIL-"):
            semantic_entity["@id"] = f"local:entity/{linked_id}"
            semantic_entity["officialName"] = f"Local Entity Group ({entity.get('entity_span', 'Unnamed')})"
            semantic_entity["description"] = f"Unmapped out-of-KB co-reference cluster {linked_id}"

        # Case C: Single Unlinked Entity Mention
        else:
            semantic_entity["@id"] = f"local:unlinked/{record.get('local_id', idx)}_{ent_idx}"
            semantic_entity["officialName"] = entity.get('entity_span', 'Unlinked Entity')
            semantic_entity["description"] = "Single occurrence unlinked textual entity extraction."

        semantic_record["entities"].append(semantic_entity)

    json_ld_document["@graph"].append(semantic_record)

# =========================================================================================================================================
# Output Serialization
# =========================================================================================================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(json_ld_document, f, indent=4)

print("\nJSON-LD EXPORT COMPLETE")
print(f"• Total Records Exported: {len(records)}")
print(f"• Total Cross-Database VIAF Links Added: {total_viaf_links}")
print(f"• Total Node Image Embeddings Embedded: {total_images_mapped}")
print(f"• Total Geographic Coordinate Nodes Mapped: {total_coordinates_mapped}")
print(f"• Total Relational Graph Edges Created: {total_paths_added}")

average_density = (total_paths_added + total_coordinates_mapped + total_images_mapped) / len(records) if records else 0
print(f"• Final Graph Visual Density Score: {average_density:.2f} paths per record.")
print(f"\nSUCCESS: Graph compiled and saved to: '{OUTPUT_FILE}'")